# Ejercicio AirBnB: Extracción de entidades

En este cuaderno vamos a trabajar con un dataset de AirBnB de la ciudad de Oporto. Se puede encontrar más información sobre el dataset y otras implementaciones en [Porto](https://github.com/Vasallo94/Porto). 

El dataset contiene información sobre las características de las viviendas, su localización, el precio, el número de comentarios, etc.

El objetivo de este ejercicio es poder analizar los comentarios de los usuarios mediante LLMs y poder extraer información relevante de los mismos para su análisis posterior.

## Primera parte

Vamos a utilizar el archivo listings1_cleaned.csv que contiene información sobre las viviendas de AirBnB en Oporto.

# 1.0 Importacion de librerias

In [12]:
import pandas as pd
from google import genai
from IPython.display import display, Markdown
import json
import os

# 1.1 Configuracion de variable de Entorno

In [13]:

from dotenv import load_dotenv

# Configurar la API Key de Gemini o configura el cliente de groq! Usa el proveedor que quieras
load_dotenv("../.env")

GOOGLE_API_KEY = os.environ["GEMINI_API_KEY"]

# 1.2 Inicializacion de Cliente Gemini

In [14]:
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# 1.3 Visualizacion de Datos

In [39]:

data = pd.read_csv("./data/listings1_cleaned.csv")
data.head

<bound method NDFrame.head of         listing_id                                       listing_url  \
0     2.905900e+04                https://www.airbnb.com/rooms/29059   
1     2.906100e+04                https://www.airbnb.com/rooms/29061   
2     3.630100e+04                https://www.airbnb.com/rooms/36301   
3     3.811800e+04                https://www.airbnb.com/rooms/38118   
4     5.047900e+04                https://www.airbnb.com/rooms/50479   
...            ...                                               ...   
8135  1.116753e+18  https://www.airbnb.com/rooms/1116753448972266015   
8136  1.116820e+18  https://www.airbnb.com/rooms/1116820020797670135   
8137  1.117028e+18  https://www.airbnb.com/rooms/1117028063118176710   
8138  1.117273e+18  https://www.airbnb.com/rooms/1117273423317641658   
8139  1.118138e+18  https://www.airbnb.com/rooms/1118138226258202248   

                                            picture_url  \
0     https://a0.muscache.com/pictures/736399/

# 1.4 Funcion de python para conseguir las 20 reseñas de mayor logitud.

In [17]:
def get_longest_descriptions(df, column="description", n=20):
    return (
        df.dropna(subset=[column])
        .assign(description_length=df[column].str.len())
        .sort_values("description_length", ascending=False)
        .head(n)
    )


top_20_descriptions = get_longest_descriptions(data, n=20)
top_20_descriptions

,listing_id,listing_url,picture_url,name,description,host_id,host_name,host_since,host_response_rate,host_acceptance_rate,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,reviews_per_month,availability_365,has_availability,last_review,geographical_group,description_length
1352,28414014.0,https://www.airbnb.com/rooms/28414014,https://a0.muscache.com/pictures/prohost-api/H...,Business Ready❤Dwtwn Conference Palais de Cong...,"Composition: bathroom, WC, bedroom / living ro...",66340481.0,Daniel,2016-04-08,97.000000,67.000000,...,4.36,4.91,4.73,5.00,0.19,167.0,t,2022-07-19,Central,1000.0
1202,25266022.0,https://www.airbnb.com/rooms/25266022,https://a0.muscache.com/pictures/prohost-api/H...,Mile End Condo with City Views from the Roof Deck,Greet the day as morning light fills a chic an...,58076590.0,Berish,2016-02-09,100.000000,99.000000,...,4.80,4.91,4.87,4.65,1.61,346.0,t,2024-02-24,Central,1000.0
2733,50852347.0,https://www.airbnb.com/rooms/50852347,https://a0.muscache.com/pictures/79e0d045-d642...,"une chambre de 20m2, idéal pour les voyageurs","Large studio next to the subway, fully furnish...",147192263.0,Yan,2017-08-22,90.000000,100.000000,...,3.97,4.59,4.54,4.64,1.82,351.0,t,2024-03-17,South,1000.0
174,2133810.0,https://www.airbnb.com/rooms/2133810,https://a0.muscache.com/pictures/917f1f75-f2c3...,joli appartement économique,Large apartment located on the island of Montr...,10873633.0,Wahab,2013-12-28,100.000000,89.000000,...,4.63,4.78,4.61,4.68,0.42,265.0,t,2023-01-02,East,1000.0
193,2524236.0,https://www.airbnb.com/rooms/2524236,https://a0.muscache.com/pictures/75cd488e-4668...,Contemporary Loft in Old Montreal Third Floor,Sit back with a freshly brewed coffee in the c...,11916402.0,Loft,2014-02-03,100.000000,100.000000,...,4.77,4.61,4.69,4.89,0.39,66.0,t,2023-09-01,Central,1000.0
690,15522103.0,https://www.airbnb.com/rooms/15522103,https://a0.muscache.com/pictures/ab30d168-c934...,"Roomy, Designer Apartment near Sherbrooke Metr...",Our apartment is located on the second floor a...,6260346.0,Adrien,2013-05-06,100.000000,87.000000,...,4.84,4.96,4.93,4.93,4.26,32.0,t,2024-03-18,Central,1000.0
729,16325175.0,https://www.airbnb.com/rooms/16325175,https://a0.muscache.com/pictures/3be5069c-8e70...,"Discover Little Italy from a Hip, Cozy Tinyhouse",Fix breakfast in a cozy kitchen and dine at a ...,5291235.0,Linda,2013-03-02,100.000000,73.000000,...,4.94,4.97,4.99,4.99,1.26,24.0,t,2024-02-20,South,1000.0
835,18508404.0,https://www.airbnb.com/rooms/18508404,https://a0.muscache.com/pictures/monet/Select-...,Luxury Historic Montreal 3 Bdrm w/Deck. # 296183,Indulge in the luxury of classic Montreal arch...,6512090.0,Nat,2013-05-21,96.922347,89.203748,...,4.96,4.97,5.00,4.95,1.37,38.0,t,2023-12-29,Central,1000.0
883,19093836.0,https://www.airbnb.com/rooms/19093836,https://a0.muscache.com/pictures/prohost-api/H...,Luxuriate in the Cozy Apartment near Mount Royal,Prep meals in the sleek kitchen with monochrom...,58076590.0,Berish,2016-02-09,100.000000,99.000000,...,4.78,4.85,4.89,4.65,1.37,347.0,t,2024-02-17,Central,1000.0
912,19717514.0,https://www.airbnb.com/rooms/19717514,https://a0.muscache.com/pictures/prohost-api/H...,"Unwind in Style at a Luxurious, Modern Condo","Kick back on the crisp, white corner sofa at t...",58076590.0,Berish,2016-02-09,100.000000,99.000000,...,4.79,4.88,4.86,4.79,2.11,352.0,t,2024-03-04,Central,1000.0


# 1.5 LLM Prompt Para devolver esas reseñas en formato JSON

In [28]:
from pydantic import BaseModel
from typing import List


class ListingEntities(BaseModel):
    name: str
    location: str
    main_characteristics: str
    type: str
    size: str
    capacity: str
    key_amenities: List[str]
    proximity_highlights: List[str]


LISTING_COLUMNS = [
    "name",
    "neighbourhood",
    "geographical_group",
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "amenities",
    "description",
]


def build_extraction_prompt(listings_df) -> str:
    listings = listings_df[LISTING_COLUMNS].fillna("No especificado").to_dict(orient="records")

    numbered_listings = "\n\n".join(
        f"""Anuncio {i + 1}:
- name (columna): {listing['name']}
- neighbourhood (columna): {listing['neighbourhood']}
- geographical_group (columna): {listing['geographical_group']}
- property_type (columna): {listing['property_type']}
- room_type (columna): {listing['room_type']}
- accommodates (columna): {listing['accommodates']}
- bedrooms (columna): {listing['bedrooms']}
- beds (columna): {listing['beds']}
- amenities (columna): {listing['amenities']}
- description (texto libre): {listing['description']}"""
        for i, listing in enumerate(listings)
    )

    return f"""Eres un asistente experto en analizar anuncios de alojamientos de Airbnb.

    Te voy a proporcionar los datos de {len(listings)} anuncios. Para cada anuncio recibirás varias columnas estructuradas del dataset (name, neighbourhood, geographical_group, property_type, room_type, accommodates, bedrooms, beds, amenities) además del texto libre de "description".

    Para CADA anuncio, extrae la siguiente información y devuélvela como un objeto JSON con exactamente estas claves:
    - "name": usa el valor de la columna "name" (str).
    - "location": combina "neighbourhood" y "geographical_group" cuando ambos estén disponibles (str).
    - "main_characteristics": resumen de 5 palabras basado en el texto de "description" (str).
    - "type": usa "property_type" y "room_type"; si no están disponibles, infiérelo del texto de "description" (str).
    - "size": usa "bedrooms" y "beds" si están disponibles; si no, infiere el tamaño a partir del texto de "description" (str).
    - "capacity": usa el valor de "accommodates"; si no está disponible, infiérelo del texto de "description" (str).
    - "key_amenities": lista de las comodidades o servicios más destacados, combinando lo que aparece en la columna "amenities" y lo que se resalta en el texto de "description". Limítate a solo las primeras 5 (list de str).
    - "proximity_highlights": lista de lugares de interés, transporte o zonas cercanas mencionadas ÚNICAMENTE en el texto de "description" (list de str).

    Reglas importantes:
    1. Prioriza siempre los datos de las columnas estructuradas cuando estén disponibles y no sean "No especificado"; usa el texto de "description" para completar o enriquecer lo que las columnas no cubran (main_characteristics, key_amenities, proximity_highlights).
    2. No inventes datos que no estén presentes ni en las columnas ni en el texto.
    3. Si un campo no se puede determinar ni por columna ni por texto, usa "No especificado" (para strings) o una lista vacía [] (para listas).
    4. Devuelve un array JSON con un objeto por cada anuncio, respetando el mismo orden en que se presentan.
    5. No incluyas texto adicional, explicaciones ni marcado Markdown; responde ÚNICAMENTE con el JSON.

    Anuncios:
    {numbered_listings}
    """


prompt = build_extraction_prompt(top_20_descriptions)
print(prompt[:1000])

Eres un asistente experto en analizar anuncios de alojamientos de Airbnb.

    Te voy a proporcionar los datos de 20 anuncios. Para cada anuncio recibirás varias columnas estructuradas del dataset (name, neighbourhood, geographical_group, property_type, room_type, accommodates, bedrooms, beds, amenities) además del texto libre de "description".

    Para CADA anuncio, extrae la siguiente información y devuélvela como un objeto JSON con exactamente estas claves:
    - "name": usa el valor de la columna "name" (str).
    - "location": combina "neighbourhood" y "geographical_group" cuando ambos estén disponibles (str).
    - "main_characteristics": resumen de 5 palabras basado en el texto de "description" (str).
    - "type": usa "property_type" y "room_type"; si no están disponibles, infiérelo del texto de "description" (str).
    - "size": usa "bedrooms" y "beds" si están disponibles; si no, infiere el tamaño a partir del texto de "description" (str).
    - "capacity": usa el valor de "

In [29]:
response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": list[ListingEntities],
        "max_output_tokens": 8192,
        "thinking_config": {"thinking_budget": 0},
    },
)

print("finish_reason:", response.candidates[0].finish_reason)
print("usage:", response.usage_metadata)

listings_json = json.loads(response.text)
listings_json

finish_reason: FinishReason.STOP
usage: cache_tokens_details=None cached_content_token_count=None candidates_token_count=3634 candidates_tokens_details=None prompt_token_count=11109 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=11109
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=14743 traffic_type=None


[{'name': 'Business Ready❤Dwtwn Conference Palais de Congres!',
  'location': 'Ville-Marie, Central',
  'main_characteristics': 'Business ready, downtown, clean, no smoking.',
  'type': 'Entire rental unit / Entire home/apt',
  'size': '1 bed',
  'capacity': '2',
  'key_amenities': ['Wifi',
   'Air conditioning',
   'Elevator',
   'Heating',
   'Kitchen'],
  'proximity_highlights': ['Downtown']},
 {'name': 'Mile End Condo with City Views from the Roof Deck',
  'location': 'Le Plateau-Mont-Royal, Central',
  'main_characteristics': 'Chic, modern, city views, hot tub.',
  'type': 'Entire condo / Entire home/apt',
  'size': '3 bedrooms, 3 beds',
  'capacity': '8',
  'key_amenities': ['Carbon monoxide alarm',
   'Microwave',
   'Crib',
   'Shampoo',
   'Hair dryer'],
  'proximity_highlights': ['Mont-Royal Park']},
 {'name': 'une chambre de 20m2, idéal pour les voyageurs',
  'location': 'Verdun, South',
  'main_characteristics': 'Large studio, furnished, quiet, clean.',
  'type': 'Private r

1. Filtra el dataset con una función que seleccione las 10/20 reseñas con mayor longitud (de string) en la columna descripción (las que consideraríamos las más 'relevantes')
2. Elabora un prompt de tal forma que un LLM sea capaz de extraer un json con las siguientes entidades de las entradas de la tabla filtrada:
```json
{
    "name": str,
    "location": str,
    "main_characteristics": str,
    "type": str,
    "size": str,
    "capacity": str,
    "key_amenities": list,
    "proximity_highlights": list,
}
```
3. Puedes elegir el proveedor que quieras, Gemini de Google o cualquiera de los modelos de Groq

## Parte 2

Realiza un análisis de los comentarios de los apartamentos con un LLM y extrae información relevante de los mismos.

Por ejemplo, puedes analizar los comentarios y extraer información sobre la limpieza, la ubicación, la relación calidad-precio, el sentimiento del comentario, etc. 

Igual que la parte 1 pero con un segundo dataset y con formato de salida a tu gusto

# 2.1 Nuevo dagtaset con Reviews

In [40]:
comentarios = pd.read_csv("./data/Airbnb_reviews_5000.csv")
comentarios

,name,host_id,host_name,date,reviewer_id,reviewer_name,comments,language
0,LUXURY apartment t3 oporto antas,37249350.0,Joaquim,2018-01-30,156292241.0,Silvia,Acomodação espaçosa e bonita. O Joaquim e sua ...,pt
1,Aida's Haven | Room&PrivateBath | St. Catarina,38365612.0,Alexandra,2019-12-15,283158910.0,Tierry Dayan,"A anfitriã Alexandra foi muito simpática, pre...",pt
2,Maritime Inspiration - One Bedroom Beach Apart...,147469727.0,Susana,2018-08-12,173756309.0,Julio,"Apartamento muy bien comunicado, aunque nos es...",es
3,Central charming Top floor - nice views,26222276.0,A.Maria,2016-05-28,11705876.0,Rafael,"Adosinda was kind, showed the apartment, set u...",en
4,Ribeira Oporto Apartment II (Renewed 2021),35057317.0,João,2016-06-02,38269559.0,Anais,"Un appartement au coeur de porto, très grand a...",fr
...,...,...,...,...,...,...,...,...
4995,Marquês's House - Estúdio ao Marquês,93991335.0,Conceição,2017-09-10,13710198.0,Sandra,Merci pour le chaleureux accueil et tous les b...,fr
4996,M1.4 | Surf Beach Matosinhos | Porto,39551356.0,Porto Je T'Aime,2018-08-04,204048434.0,Louis,Toutes les qualités recherchées dans ce type d...,fr
4997,Kitchnette Studio in Porto's Downtown II,1651078.0,Lurdes,2022-06-19,68564036.0,Angie,"Its a lovely little studio, a green oasis, rig...",en
4998,MyLoft-Charming Apartment!!! - Metro Bolhão,34348897.0,Carla,2022-06-23,283375850.0,Angelika,Carla ist wirklich eine sehr tolle Gastgeberin...,de


# 2.2 Selección de las comentarios más relevantes

Reutilizamos `get_longest_descriptions` (definida en la parte 1) sobre la columna `comments` para quedarnos con las 20 comentarios más largos, que asumimos como las más ricas en información.

In [36]:
top_20_comments = get_longest_descriptions(comentarios, column="comments", n=20)
top_20_comments

,name,host_id,host_name,date,reviewer_id,reviewer_name,comments,language,description_length
2000,FLH Porto Homey Flat with Balcony,3953109.0,Feels Like Home,2022-10-27,8576401.0,Ricardo,This WOULD have been a 5 star review BUT:<br/>...,en,2976.0
3876,Oporto FR Cativo Flat,27518920.0,Fábio,2015-05-18,11194726.0,Nicola,Two of us stayed in the apartment for a week i...,en,2392.0
2456,Terrace Duplex in Porto,136571788.0,João,2017-10-12,20110201.0,Elizeane,"The location of the flat is great, few minutes...",en,2226.0
1908,Belomonte River View Apartments 1,79861584.0,Belomonte River View Apartments,2022-06-29,103620613.0,Matthew,Amazing views of the Douro River with Porto on...,en,2011.0
1636,Pinheiro Cardoso House - Room 4,47583333.0,Luís,2022-10-09,401099365.0,Maria,Não recomendo! De todo! <br/>Este Airbnb é no ...,pt,1928.0
205,Marias River Apartment,250086123.0,Nuno,2022-05-25,409871736.0,Sabrina,Fomos surpreendidos por um delicioso vinho do ...,pt,1846.0
1963,New! La vie en rose by Upperground,101542522.0,Upperground,2022-11-04,9079292.0,Shirin,The apartment was alright for its price range....,en,1817.0
4992,Casas Brancas w/ Balcony & free private parkin...,280564100.0,LovelyStay,2022-08-12,2620519.0,Juan,Apartamento bien distribuido y bien surtido. M...,es,1808.0
4707,GuestReady - Beige Glamour,364783572.0,GuestReady,2017-06-22,129290118.0,Tom,On visitait Porto pour la première fois avec m...,fr,1795.0
1957,Feel Porto Doc Townhouse,2228036.0,Rui,2019-05-13,45966983.0,Lynnda,We very much enjoyed our stay at Rui’s apartme...,en,1781.0


# 2.3 LLM Prompt para analizar las reseñas

Las reseñas están escritas en distintos idiomas (portugués, español, inglés, francés, alemán...). Pedimos al LLM que las interprete en su idioma original pero que devuelva siempre el análisis en español, extrayendo sentimiento, limpieza, ubicación, relación calidad-precio y trato del anfitrión.

In [37]:
class ReviewAnalysis(BaseModel):
    sentiment: str
    cleanliness: str
    location: str
    value_for_money: str
    host_experience: str
    summary: str


def build_review_analysis_prompt(reviews_df) -> str:
    reviews = reviews_df[["name", "language", "comments"]].to_dict(orient="records")

    numbered_reviews = "\n\n".join(
        f"""Reseña {i + 1} (alojamiento: {review['name']}, idioma original: {review['language']}):
{review['comments']}"""
        for i, review in enumerate(reviews)
    )

    return f"""Eres un asistente experto en analizar reseñas de huéspedes de alojamientos de Airbnb escritas en distintos idiomas (portugués, español, inglés, francés, alemán, etc.).

    Te voy a proporcionar {len(reviews)} reseñas. Cada una indica el alojamiento al que pertenece y su idioma original; interprétala en ese idioma pero responde SIEMPRE en español.

    Para CADA reseña, extrae la siguiente información y devuélvela como un objeto JSON con exactamente estas claves:
    - "sentiment": sentimiento general de la reseña. Debe ser uno de: "positivo", "negativo", "neutral" o "mixto" (str).
    - "cleanliness": resumen de lo que se dice sobre la limpieza del alojamiento, o "No mencionado" si no se menciona (str).
    - "location": resumen de lo que se dice sobre la ubicación del alojamiento, o "No mencionado" si no se menciona (str).
    - "value_for_money": resumen de lo que se dice sobre la relación calidad-precio, o "No mencionado" si no se menciona (str).
    - "host_experience": resumen de lo que se dice sobre el trato, la comunicación o la atención del anfitrión, o "No mencionado" si no se menciona (str).
    - "summary": resumen general de la reseña en español, en una frase de máximo 15 palabras (str).

    Reglas importantes:
    1. Traduce e interpreta el contenido aunque la reseña no esté en español; el JSON de salida debe estar siempre en español.
    2. Usa EXCLUSIVAMENTE información presente en el texto; no inventes datos que no se puedan inferir razonablemente de la reseña.
    3. Si un aspecto no se menciona, usa "No mencionado" en el campo correspondiente.
    4. Devuelve un array JSON con un objeto por cada reseña, respetando el mismo orden en que se presentan.
    5. No incluyas texto adicional, explicaciones ni marcado Markdown; responde ÚNICAMENTE con el JSON.

    Reseñas:
    {numbered_reviews}
    """


review_prompt = build_review_analysis_prompt(top_20_comments)
print(review_prompt[:1000])

Eres un asistente experto en analizar reseñas de huéspedes de alojamientos de Airbnb escritas en distintos idiomas (portugués, español, inglés, francés, alemán, etc.).

    Te voy a proporcionar 20 reseñas. Cada una indica el alojamiento al que pertenece y su idioma original; interprétala en ese idioma pero responde SIEMPRE en español.

    Para CADA reseña, extrae la siguiente información y devuélvela como un objeto JSON con exactamente estas claves:
    - "sentiment": sentimiento general de la reseña. Debe ser uno de: "positivo", "negativo", "neutral" o "mixto" (str).
    - "cleanliness": resumen de lo que se dice sobre la limpieza del alojamiento, o "No mencionado" si no se menciona (str).
    - "location": resumen de lo que se dice sobre la ubicación del alojamiento, o "No mencionado" si no se menciona (str).
    - "value_for_money": resumen de lo que se dice sobre la relación calidad-precio, o "No mencionado" si no se menciona (str).
    - "host_experience": resumen de lo que se d

In [38]:
review_response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=review_prompt,
    config={
        "response_mime_type": "application/json",
        "response_schema": list[ReviewAnalysis],
        "max_output_tokens": 8192,
        "thinking_config": {"thinking_budget": 0},
    },
)

print("finish_reason:", review_response.candidates[0].finish_reason)
print("usage:", review_response.usage_metadata)

reviews_json = json.loads(review_response.text)
reviews_json

finish_reason: FinishReason.STOP
usage: cache_tokens_details=None cached_content_token_count=None candidates_token_count=1614 candidates_tokens_details=None prompt_token_count=9313 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=9313
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=10927 traffic_type=None


[{'sentiment': 'mixto',
  'cleanliness': 'No mencionado',
  'location': 'No mencionado',
  'value_for_money': 'negativo',
  'host_experience': 'negativo',
  'summary': 'El apartamento es bonito, pero la gestión y mantenimiento son deficientes, restando valor.'},
 {'sentiment': 'positivo',
  'cleanliness': 'positivo',
  'location': 'positivo',
  'value_for_money': 'No mencionado',
  'host_experience': 'positivo',
  'summary': 'Apartamento bien ubicado, reformado y anfitrión atento, superando expectativas.'},
 {'sentiment': 'mixto',
  'cleanliness': 'No mencionado',
  'location': 'positivo',
  'value_for_money': 'No mencionado',
  'host_experience': 'positivo',
  'summary': 'Excelente ubicación y anfitrión, pero faltan utensilios y el sofá cama es incómodo.'},
 {'sentiment': 'positivo',
  'cleanliness': 'No mencionado',
  'location': 'positivo',
  'value_for_money': 'No mencionado',
  'host_experience': 'positivo',
  'summary': 'Vistas increíbles y buena ubicación, con anfitriona atenta 

,name,host_id,host_name,date,reviewer_id,reviewer_name,comments,language
0,LUXURY apartment t3 oporto antas,37249350.0,Joaquim,2018-01-30,156292241.0,Silvia,Acomodação espaçosa e bonita. O Joaquim e sua ...,pt
1,Aida's Haven | Room&PrivateBath | St. Catarina,38365612.0,Alexandra,2019-12-15,283158910.0,Tierry Dayan,"A anfitriã Alexandra foi muito simpática, pre...",pt
2,Maritime Inspiration - One Bedroom Beach Apart...,147469727.0,Susana,2018-08-12,173756309.0,Julio,"Apartamento muy bien comunicado, aunque nos es...",es
3,Central charming Top floor - nice views,26222276.0,A.Maria,2016-05-28,11705876.0,Rafael,"Adosinda was kind, showed the apartment, set u...",en
4,Ribeira Oporto Apartment II (Renewed 2021),35057317.0,João,2016-06-02,38269559.0,Anais,"Un appartement au coeur de porto, très grand a...",fr
...,...,...,...,...,...,...,...,...
4995,Marquês's House - Estúdio ao Marquês,93991335.0,Conceição,2017-09-10,13710198.0,Sandra,Merci pour le chaleureux accueil et tous les b...,fr
4996,M1.4 | Surf Beach Matosinhos | Porto,39551356.0,Porto Je T'Aime,2018-08-04,204048434.0,Louis,Toutes les qualités recherchées dans ce type d...,fr
4997,Kitchnette Studio in Porto's Downtown II,1651078.0,Lurdes,2022-06-19,68564036.0,Angie,"Its a lovely little studio, a green oasis, rig...",en
4998,MyLoft-Charming Apartment!!! - Metro Bolhão,34348897.0,Carla,2022-06-23,283375850.0,Angelika,Carla ist wirklich eine sehr tolle Gastgeberin...,de
